In [3]:
import json
from pathlib import Path

from sentence_transformers import SentenceTransformer
import numpy as np

In [30]:
language = "mandan"

filename = f"../data/sections/{language}_sections.json"

with open(filename, encoding="utf-8") as f:
    data = json.load(f)

print(len(data))

# Get language name from filename
language_name = Path(filename).stem.split("_")[0]

276


In [31]:
# Remove empty sections
data = [section for section in data if section["text"].strip() != ""]
print(len(data))

276


In [32]:
for s in data:
    heading_level = s["heading_level"]
    if heading_level == 1:
        s["parent_headings"] = None
    else:
        # Find all the parent heading until level 1
        
        parent_headings = []
        for level in range(heading_level - 1, 0, -1):
            # Find the nearest heading with this level before the current section
            for prev_s in reversed(data[: data.index(s)]):
                if prev_s["heading_level"] == level:
                    parent_headings.append(prev_s["heading"])
                    break
        if parent_headings:
            s["parent_headings"] = parent_headings[::-1]  # Reverse to get from top to bottom
        else:
            s["parent_headings"] = None


In [33]:
for s in data:
    if s["heading_level"] != 1:
        print(
            f"Level {s['heading_level']} Heading: {s['heading']}, Parent: {s['parent_headings']}")

Level 0 Heading: A grammar of Mandan, Parent: None
Level 4 Heading: Comprehensive Grammar Library, Parent: None
Level 0 Heading: A grammar of Mandan, Parent: None
Level 2 Heading: 1.1 Background on the Mandan people, Parent: ['Abbreviations 1 Introduction']
Level 3 Heading: 1.1.1 Overview, Parent: ['Abbreviations 1 Introduction', '1.1 Background on the Mandan people']
Level 3 Heading: 1.1.2 900 CE to 1851 CE, Parent: ['Abbreviations 1 Introduction', '1.1 Background on the Mandan people']
Level 3 Heading: 1.1.3 1851 CE to present day, Parent: ['Abbreviations 1 Introduction', '1.1 Background on the Mandan people']
Level 2 Heading: 1.2 Genetic relationships, Parent: ['Abbreviations 1 Introduction']
Level 2 Heading: 1.3 Previous research on the Mandan language, Parent: ['Abbreviations 1 Introduction']
Level 2 Heading: 1.4 Personal fieldwork and sources of data, Parent: ['Abbreviations 1 Introduction']
Level 3 Heading: 1.4.1 Personal fieldwork, Parent: ['Abbreviations 1 Introduction', '1.4 

In [34]:
# Embed using all-MiniLM-L6-v2
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    [f"{section['heading']} {section['text']}" for section in data],
    show_progress_bar=True,
    convert_to_numpy=True,
)

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

In [35]:
heading_text = []
for section in data:
    if section["parent_headings"]:
        heading_text.append(" ".join(section["parent_headings"] + [section["heading"]]))
    else:
        heading_text.append(section["heading"])

heading_embeddings = model.encode(
    heading_text,
    show_progress_bar=True,
    convert_to_numpy=True,
)

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

In [36]:
# Simple embedding similarity with words

classes = {

    # REMOVE ENTIRELY
    "structure": [
        "contents", "appendix", "references", "bibliography", "index",
        "maps", "figures", "tables"
    ],

    "metadata": [
        "title", "author", "abstract", "acknowledgements",
        "abbreviations", "publisher", "copyright", "edition",
        "notation", "glossing", "method", "data", "transcription"
    ],

    "background": [
        "history", "historical", "development", "genealogy",
        "sociolinguistics", "ethnographic", "geography", "community",
        "settlement", "contact", "culture"
    ],

    "narratives": [
        "text", "story", "narrative", "myth",
        "glossed text", "free translation"
    ],

    # KEEP FOR RULE EXTRACTION
    "phonology": [
        "phonology", "phonetics", "sounds", "pronunciation",
        "phonemes", "allophones", "prosody", "stress", "intonation",
        "phonotactics", "syllable"
    ],

    "morphology": [
        "morphology", "morphemes", "inflection", "derivation",
        "word formation", "reduplication", "affix", "clitic",
        "nominalisation", "compounding"
    ],

    "syntax": [
        "syntax", "sentence structure", "word order", "clauses",
        "valency", "argument structure", "serial verb",
        "coordination", "negation", "interrogative", "imperative",
        "predicate", "relative clause", "non-verbal clause",
        "possessive construction", "quantifier clauses",
        "postpositions", "pronouns", "demonstratives"
    ],

    "morphosyntax": [
        "agreement", "alignment", "case marking",
        "role marking", "np structure", "nominal structure",
        "noun phrase", "determiner", "adnominal"
    ],

    "discourse": [
        "information structure", "focus", "topic",
        "evidentiality", "discourse", "narrative structure",
        "formulaic expressions", "interjections", "ideophones"
    ],

    # NEUTRAL/OPTIONAL
    "lexicon": [
        "vocabulary", "glossary", "lexicon", "wordlist", "corpus"
    ]
}


In [37]:
texts = {key: " ".join(value) for key, value in classes.items()}

class_embeddings = model.encode(
    [text for text in texts.values()],
    show_progress_bar=True,
    convert_to_numpy=True,
)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [38]:

similarity_matrix = np.inner(embeddings, class_embeddings)
predicted_classes = np.argmax(similarity_matrix, axis=1)
for section, class_idx in zip(data, predicted_classes):
    section["predicted_class"] = list(texts.keys())[class_idx]
    

In [39]:
for s in data:
    print(s["predicted_class"], ":", s["text"][:60].replace("\n", " "), "...")

syntax : Ryan M. KasakComprehensive Grammar Library 10 ...
morphology : Editor: Martin Haspelmath In this series:1. Jacques, Guillau ...
morphology : Ryan M. KasakRyan M. Kasak. 2024. A grammar of Mandan (Compr ...
structure : 5 ...
background : First and foremost, I want to expressly state that I profess ...
narratives : Núu'etaa numá'kaaki wáakapusmak wakápusa wakų́'ro'sh .This b ...
morphology : Mandan [ISO: mhq] is a Siouan language spoken in northwester ...
background : In this section, I provide a background of the Mandan people ...
background : According to tradition, the Mandan have several creation sto ...
background : The Mandan and their ancestors have lived near the Middle Mi ...
background : The government included the Three Affiliated Tribes in with  ...
background : The position of Mandan within the Siouan language family has ...
background : This section serves to examine the research on the Mandan la ...
lexicon : This section serves to explain the conditions under whi

In [40]:
similarity_matrix_heading = np.inner(heading_embeddings, class_embeddings)
predicted_classes_heading = np.argmax(similarity_matrix_heading, axis=1)
for section, class_idx in zip(data, predicted_classes_heading):
    section["predicted_class_heading"] = list(texts.keys())[class_idx]

In [41]:
for s in data:
    print(s["predicted_class_heading"], ":", s["heading"], s["parent_headings"])

syntax : A grammar of Mandan None
lexicon : Comprehensive Grammar Library None
syntax : A grammar of Mandan None
structure : Contents None
metadata : Preface None
metadata : Acknowledgments None
metadata : Abbreviations 1 Introduction None
metadata : 1.1 Background on the Mandan people ['Abbreviations 1 Introduction']
background : 1.1.1 Overview ['Abbreviations 1 Introduction', '1.1 Background on the Mandan people']
background : 1.1.2 900 CE to 1851 CE ['Abbreviations 1 Introduction', '1.1 Background on the Mandan people']
background : 1.1.3 1851 CE to present day ['Abbreviations 1 Introduction', '1.1 Background on the Mandan people']
metadata : 1.2 Genetic relationships ['Abbreviations 1 Introduction']
morphology : 1.3 Previous research on the Mandan language ['Abbreviations 1 Introduction']
metadata : 1.4 Personal fieldwork and sources of data ['Abbreviations 1 Introduction']
metadata : 1.4.1 Personal fieldwork ['Abbreviations 1 Introduction', '1.4 Personal fieldwork and sources of d

In [42]:
filtered_sections = []
for s_idx,s in enumerate(data):

    if s_idx < 12:
        print(s["predicted_class_heading"], ":", s["heading"], s["parent_headings"])
        s["filtered"] = True
        continue
    if s["predicted_class_heading"] not in ["phonology", "structure", "metadata", "background", "narrative"]:
        filtered_sections.append(s)
        s["filtered"] = False
    else:
        s["filtered"] = True

print(f"Filtered sections from {len(data)} to {len(filtered_sections)}")

syntax : A grammar of Mandan None
lexicon : Comprehensive Grammar Library None
syntax : A grammar of Mandan None
structure : Contents None
metadata : Preface None
metadata : Acknowledgments None
metadata : Abbreviations 1 Introduction None
metadata : 1.1 Background on the Mandan people ['Abbreviations 1 Introduction']
background : 1.1.1 Overview ['Abbreviations 1 Introduction', '1.1 Background on the Mandan people']
background : 1.1.2 900 CE to 1851 CE ['Abbreviations 1 Introduction', '1.1 Background on the Mandan people']
background : 1.1.3 1851 CE to present day ['Abbreviations 1 Introduction', '1.1 Background on the Mandan people']
metadata : 1.2 Genetic relationships ['Abbreviations 1 Introduction']
Filtered sections from 276 to 207


In [43]:
# Store tagged sections
output_filename = f"../data/section_tagged/{language}_sections_classified.json"
with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

print(f"Saved classified sections to {output_filename}")

Saved classified sections to ../data/section_tagged/mandan_sections_classified.json
